# Conjugation boxes: Pauli gadgets and two-term LCU

**Download Notebook** - {nb-download}`conjugation_box.ipynb`

A conjugation box applies a change of basis, a middle operation, and the inverse
change of basis. This notebook shows how to build one with `ConjugationBox`
and use its ordinary, daggered, controlled, and controlled-daggered forms.

We will build two examples:

- An XYZ Pauli gadget, with a basis change for the Y operator.
- A two-term LCU block encoding, with Ry preparation and controlled-Pauli SELECT.

Run the cells from top to bottom. From a source checkout, install the notebook
and simulation dependencies with `uv sync --extra dev-dependencies`.

In [1]:
import numpy as np
import zixy.qubit.pauli as zqp
from scipy.linalg import expm

from guppylang import guppy
from guppylang.std.angles import angle
from guppylang.std.builtins import array, control, dagger, output
from guppylang.std.quantum import (
    collect_measurements, cx, cz, h, measure_array, qubit, ry, rz, sdg, x,
)

from guppyalgos.primitives.subroutines.conjugation import ConjugationBox
from guppyalgos.testing import assert_unitary_modifiers
from guppyalgos.utils import qarray

# Guppy angles are in half-turns: the angle in radians is alpha = pi * theta.
theta = 0.37
alpha = np.pi * theta

## 1. What does the box do?

The fields `g` and `u` store Guppy functions acting on the same register.
G must be daggerable; U must support both dagger and control.

The code executes **G, then U, then G†**. Matrix products act right to left,
so the resulting unitary is

$$
W = G^{\dagger} U G,
\qquad
W^{\dagger} = G^{\dagger} U^{\dagger} G.
$$

Create the box inside a Guppy function and call `box.compose(qs)`.
`compose` is an inner `@guppy.unitary` class; the surrounding modifier selects
its implementation:

| Call context | Execution order |
| --- | --- |
| Ordinary call | G → U → G† |
| `with dagger:` | G → U† → G† |
| `with control(controls):` | G → controlled-U → G† |
| `with control(controls), dagger:` | G → controlled-U† → G† |

For an external control c,

$$
\mathrm{C}(W)
= |0\rangle\langle0|_c \otimes I
+ |1\rangle\langle1|_c \otimes W
= (I_c\otimes G^{\dagger})\,\mathrm{C}(U)\,(I_c\otimes G).
$$

Only U receives the external control. G and G† run in both branches. When c is
zero, the middle operation is the identity, so G† G cancels. When c is one,
the box applies W. The same argument applies to U† in a controlled dagger call.

## 2. An XYZ Pauli gadget

Let $P = X_0Y_1Z_2$. Use Hadamard on q0 and the sequence S†, H on q1 to
change to the Z basis. Two CNOTs collect the parity onto q2. Together these
operations form G, with

$$
G^{\dagger} Z_2 G = X_0Y_1Z_2.
$$

The middle operation is $U=R_z(\alpha)$ on q2, where $\alpha=\pi\theta$.
The complete gadget is

$$
W_{XYZ}(\theta)
= G^{\dagger}R_z(\alpha)G
= e^{-i\alpha X_0Y_1Z_2/2}.
$$

The preparation function below needs only `daggerable=True`. The rotation is
unitary so that it can be controlled and daggered.

In [2]:
@guppy(daggerable=True)
def g(qs: array[qubit, 3]) -> None:
    h(qs[0])
    sdg(qs[1])
    h(qs[1])
    cx(qs[0], qs[2])
    cx(qs[1], qs[2])


@guppy(unitary=True)
def u(qs: array[qubit, 3]) -> None:
    rz(qs[2], angle(theta))

### Call, dagger, control, and controlled dagger

The following program calls the box in all four modes. It prepares the external
control in a superposition so both control branches participate.

The ordinary call followed by its dagger cancels. The controlled call followed
by its controlled dagger also cancels. After undoing the initial Hadamards,
both measured registers should contain only zeros.

For this gadget, the middle controlled operation is

$$
\mathrm{C}(U)=\mathrm{C}R_z(\alpha)
$$

on q2. The external control is not attached to the Hadamards, S†, or parity
CNOTs in G or G†.

In [3]:
@guppy
def run_pauli_gadget() -> None:
    qs = qarray(3)
    controls = qarray(1)
    h(qs[0])
    h(controls[0])
    box = ConjugationBox(g, u)

    box.compose(qs)
    with dagger:
        box.compose(qs)
    with control(controls):
        box.compose(qs)
    with control(controls), dagger:
        box.compose(qs)

    h(qs[0])
    h(controls[0])
    output("targets", collect_measurements(measure_array(qs)))
    output("control", collect_measurements(measure_array(controls)))


pauli_counts = (
    run_pauli_gadget.emulator(4)
    .with_seed(42)
    .with_shots(8)
    .run()
    .collated_counts()
)
print(pauli_counts)

Counter({(('targets', '000'), ('control', '0')): 8})


### Verify the complete operation

Build an independent reference from a Zixy Pauli Hamiltonian. Big-endian matrix
ordering matches the default used by `assert_unitary_modifiers`.

The small unitary wrapper below lets the helper apply each modifier to the same
call. It checks the forward matrix against $e^{-i\alpha P/2}$ and checks all
three modified forms, including the identity branch of the external control.

In [4]:
@guppy(unitary=True)
def pauli_gadget(qs: array[qubit, 3]) -> None:
    box = ConjugationBox(g, u)
    box.compose(qs)


pauli_hamiltonian = zqp.RealTermSum.from_str("(1.0, X0 Y1 Z2)", 3)
pauli_matrix = pauli_hamiltonian.to_sparse_matrix(big_endian=True).toarray()
pauli_reference = expm(-0.5j * alpha * pauli_matrix)

assert_unitary_modifiers(pauli_gadget, 3, expected_unitary=pauli_reference)
print("XYZ gadget: all four modifier modes match the reference matrices.")

XYZ gadget: all four modifier modes match the reference matrices.


## 3. A two-term LCU block encoding

Now use a two-qubit register: `qs[0]` is the index qubit a and `qs[1]` is the
target t. PREPARE rotates only the index qubit:

$$
\mathrm{PREPARE}=R_y(\alpha)_a\otimes I_t,
\qquad
R_y(\alpha)|0\rangle
= c_0|0\rangle+c_1|1\rangle,
$$

where $c_0=\cos(\alpha/2)$ and $c_1=\sin(\alpha/2)$.

SELECT applies X when the index is zero and Z when the index is one:

$$
\mathrm{SELECT}
= |0\rangle\langle0|_a\otimes X_t
+ |1\rangle\langle1|_a\otimes Z_t.
$$

We set G = PREPARE and U = SELECT. The box implements the full unitary

$$
B=\mathrm{PREPARE}^{\dagger}\,\mathrm{SELECT}\,\mathrm{PREPARE}.
$$

Its index-zero block is the two-term LCU:

$$
(\langle0|_a\otimes I_t)\,B\,(|0\rangle_a\otimes I_t)
= c_0^2 X_t+c_1^2 Z_t = A.
$$

A need not be unitary. B is the unitary block encoding and retains the index
qubit throughout the examples and modifier checks.

In [5]:
@guppy(daggerable=True)
def prepare(qs: array[qubit, 2]) -> None:
    ry(qs[0], angle(theta))


@guppy(unitary=True)
def select(qs: array[qubit, 2]) -> None:
    # Open-index control: apply X to the target when the index is zero.
    x(qs[0])
    cx(qs[0], qs[1])
    x(qs[0])
    # Closed-index control: apply Z to the target when the index is one.
    cz(qs[0], qs[1])

### Distinguish the index from the external control

The index a already controls which Pauli SELECT applies. An external control c
has a different role: it turns the whole SELECT operation on or off.

$$
\mathrm{C}(B)
= (I_c\otimes\mathrm{PREPARE}^{\dagger})
\,\mathrm{C}(\mathrm{SELECT})\,
(I_c\otimes\mathrm{PREPARE}).
$$

- When c is zero, SELECT is skipped and PREPARE† PREPARE cancels.
- When c is one, SELECT uses a to choose X or Z, giving B.
- PREPARE and PREPARE† remain uncontrolled in both cases.

Controlling SELECT controls its complete implementation, including the X gates
that implement the open-index control. The outer control therefore acts on the
whole block encoding B, rather than on the nonunitary projected matrix A alone.

In [6]:
@guppy
def run_lcu() -> None:
    qs = qarray(2)
    controls = qarray(1)
    h(qs[1])
    h(controls[0])
    box = ConjugationBox(prepare, select)

    box.compose(qs)
    with dagger:
        box.compose(qs)
    with control(controls):
        box.compose(qs)
    with control(controls), dagger:
        box.compose(qs)

    h(qs[1])
    h(controls[0])
    output("index_and_target", collect_measurements(measure_array(qs)))
    output("control", collect_measurements(measure_array(controls)))


lcu_counts = (
    run_lcu.emulator(3)
    .with_seed(42)
    .with_shots(8)
    .run()
    .collated_counts()
)
print(lcu_counts)

Counter({(('index_and_target', '00'), ('control', '0')): 8})


### Verify the unitary and its LCU block

Construct PREPARE and SELECT independently with Zixy. Expanding the index
projectors gives

$$
\mathrm{SELECT}
= \tfrac12\left(X_t+Z_aX_t+Z_t-Z_aZ_t\right).
$$

We check all four modifier modes of B, then compare its top-left block with
$c_0^2X+c_1^2Z$. Here SELECT is self-adjoint, so B is also self-adjoint;
the dagger and controlled-dagger calls still use the same modifier interface.

In [7]:
@guppy(unitary=True)
def lcu(qs: array[qubit, 2]) -> None:
    box = ConjugationBox(prepare, select)
    box.compose(qs)


prepare_hamiltonian = zqp.RealTermSum.from_str("(1.0, Y0)", 2)
prepare_matrix = expm(
    -0.5j * alpha * prepare_hamiltonian.to_sparse_matrix(big_endian=True).toarray()
)
select_hamiltonian = zqp.RealTermSum.from_str(
    "(0.5, X1), (0.5, Z0 X1), (0.5, Z1), (-0.5, Z0 Z1)", 2
)
select_matrix = select_hamiltonian.to_sparse_matrix(big_endian=True).toarray()
lcu_reference = prepare_matrix.conj().T @ select_matrix @ prepare_matrix

assert_unitary_modifiers(lcu, 2, expected_unitary=lcu_reference)

weight_x = np.cos(alpha / 2) ** 2
weight_z = np.sin(alpha / 2) ** 2
lcu_hamiltonian = zqp.RealTermSum.from_str(
    f"({weight_x}, X0), ({weight_z}, Z0)", 1
)
expected_block = lcu_hamiltonian.to_sparse_matrix(big_endian=True).toarray()
np.testing.assert_allclose(lcu_reference[:2, :2], expected_block, atol=1e-10)

print(f"LCU block: {weight_x:.6f} X + {weight_z:.6f} Z")
print("LCU: all four modifier modes and the projected block match.")

LCU block: 0.698574 X + 0.301426 Z
LCU: all four modifier modes and the projected block match.


## 4. Try another angle

Change `theta` in the setup cell and rerun the notebook. The circuits and their
reference matrices use the same input angle.

For both examples, the modifier syntax stays the same. What changes is the
middle operation: a controlled Rz for the Pauli gadget, and a controlled SELECT
for the LCU block encoding. G and G† continue to run unconditionally.